In [1]:
from pathlib import Path

# ==============================
# PROJECT CONFIGURATION
# ==============================

PROJECT_DIR = Path("/home/manish/trading_project/stock_screener")

# Existing FYERS credentials
ROOT_DIR = Path("/home/manish/trading_project")

ENV_FILE = ROOT_DIR / ".env"
ACCESS_FILE = ROOT_DIR / "access.txt"

# Scanner condition
REFERENCE_PRICE = 500.0
RANGE_PERCENT = 5.0

LOWER_PRICE = REFERENCE_PRICE * (1 - RANGE_PERCENT / 100)
UPPER_PRICE = REFERENCE_PRICE * (1 + RANGE_PERCENT / 100)

print("Reference Price :", REFERENCE_PRICE)
print("Range           :", f"{LOWER_PRICE:.2f} - {UPPER_PRICE:.2f}")
print("Access File     :", ACCESS_FILE)

Reference Price : 500.0
Range           : 475.00 - 525.00
Access File     : /home/manish/trading_project/access.txt


In [2]:
from pathlib import Path

# ==============================
# PROJECT CONFIGURATION
# ==============================

PROJECT_DIR = Path("/home/manish/trading_project/stock_screener")

# Existing FYERS credentials
ROOT_DIR = Path("/home/manish/trading_project")

ENV_FILE = ROOT_DIR / ".env"
ACCESS_FILE = ROOT_DIR / "access.txt"

# Scanner condition
REFERENCE_PRICE = 500.0
RANGE_PERCENT = 5.0

LOWER_PRICE = REFERENCE_PRICE * (1 - RANGE_PERCENT / 100)
UPPER_PRICE = REFERENCE_PRICE * (1 + RANGE_PERCENT / 100)

print("Reference Price :", REFERENCE_PRICE)
print("Range           :", f"{LOWER_PRICE:.2f} - {UPPER_PRICE:.2f}")
print("Access File     :", ACCESS_FILE)

Reference Price : 500.0
Range           : 475.00 - 525.00
Access File     : /home/manish/trading_project/access.txt


In [3]:
# ==========================================
# CELL 2 — FYERS API CLIENT
# ==========================================

import os
from dotenv import load_dotenv
from fyers_apiv3 import fyersModel

# Load existing FYERS configuration
load_dotenv(ENV_FILE)

APP_ID = os.getenv("FYERS_APP_ID")

if not APP_ID:
    raise ValueError("FYERS_APP_ID not found in .env")

# Load existing access token
ACCESS_TOKEN = ACCESS_FILE.read_text().strip()

if not ACCESS_TOKEN:
    raise ValueError("FYERS access token is empty")

# Create FYERS API client
fyers = fyersModel.FyersModel(
    client_id=APP_ID,
    token=ACCESS_TOKEN,
    log_path=""
)

print("FYERS client created successfully.")
print("App ID:", APP_ID)
print("Access token: loaded")

FYERS client created successfully.
App ID: 4DK6EK1V8X-100
Access token: loaded


In [4]:
# Check existing FYERS configuration files
from pathlib import Path

print(".env exists:", ENV_FILE.exists())
print("access.txt exists:", ACCESS_FILE.exists())

if ENV_FILE.exists():
    print("\n.env contents (variable names only):")
    for line in ENV_FILE.read_text().splitlines():
        line = line.strip()
        if line and not line.startswith("#") and "=" in line:
            print(line.split("=", 1)[0].strip())

print("\naccess.txt:")
print("exists:", ACCESS_FILE.exists())
print("size:", ACCESS_FILE.stat().st_size if ACCESS_FILE.exists() else 0)

.env exists: True
access.txt exists: True

.env contents (variable names only):
FYERS_APP_ID
FYERS_SECRET_KEY
FYERS_REDIRECT_URI

access.txt:
exists: True
size: 652


In [5]:
# ==========================================
# CELL 3 — TEST LIVE LTP
# ==========================================

test_symbols = [
    "NSE:RELIANCE-EQ",
    "NSE:TCS-EQ",
    "NSE:INFY-EQ",
]

data = {
    "symbols": ",".join(test_symbols)
}

response = fyers.quotes(data=data)

print(response)

{'message': '', 'code': 200, 'd': [{'n': 'NSE:RELIANCE-EQ', 'v': {'ask': 1248, 'bid': 0, 'chp': 0.61, 'ch': 7.6, 'description': 'NSE:RELIANCE-EQ', 'exchange': 'NSE', 'fyToken': '10100000002885', 'high_price': 1252.8, 'low_price': 1238.9, 'lp': 1248, 'open_price': 1242, 'original_name': 'NSE:RELIANCE-EQ', 'prev_close_price': 1248, 'short_name': 'RELIANCE-EQ', 'spread': 0, 'symbol': 'NSE:RELIANCE-EQ', 'tt': '1790208000', 'volume': 8352048, 'atp': 1247.3, 'imbalance': 0, 'mkt_imbalance': 0, 'reference_price': 0, 'indicative_price': 0}, 's': 'ok'}, {'n': 'NSE:TCS-EQ', 'v': {'ask': 2089.6, 'bid': 0, 'chp': -0.73, 'ch': -15.4, 'description': 'NSE:TCS-EQ', 'exchange': 'NSE', 'fyToken': '101000000011536', 'high_price': 2104.2, 'low_price': 2077.6, 'lp': 2089.6, 'open_price': 2099, 'original_name': 'NSE:TCS-EQ', 'prev_close_price': 2089.6, 'short_name': 'TCS-EQ', 'spread': 0, 'symbol': 'NSE:TCS-EQ', 'tt': '1790208000', 'volume': 2228402, 'atp': 2087.81, 'imbalance': 0, 'mkt_imbalance': 0, 'refe

In [6]:
# ==========================================
# CELL 4 — MULTI PRICE LEVEL SCANNER
# ==========================================

import pandas as pd

# Price levels to scan
PRICE_LEVELS = [100, 500, 1000, 1500, 2000, 3000, 5000]

RANGE_PERCENT = 5.0

results = []

# Scan each reference price level
for reference_price in PRICE_LEVELS:

    lower_price = reference_price * (1 - RANGE_PERCENT / 100)
    upper_price = reference_price * (1 + RANGE_PERCENT / 100)

    # Check all stocks returned by FYERS
    for item in response["d"]:

        symbol = item["n"]
        quote = item["v"]

        ltp = quote.get("lp")

        if ltp is None:
            continue

        # Check whether LTP is within ±5% of reference price
        if lower_price <= ltp <= upper_price:

            distance_percent = (
                (ltp - reference_price) / reference_price
            ) * 100

            results.append({
                "Symbol": symbol,
                "LTP": ltp,
                "Reference": reference_price,
                "Range Low": round(lower_price, 2),
                "Range High": round(upper_price, 2),
                "Distance (%)": round(distance_percent, 2),
                "Change (%)": quote.get("chp"),
                "Volume": quote.get("volume")
            })

scanner_df = pd.DataFrame(results)

if scanner_df.empty:
    print("No stocks found near the selected price levels.")

else:
    scanner_df = scanner_df.sort_values(
        ["Reference", "Distance (%)"]
    ).reset_index(drop=True)

    display(scanner_df)

,Symbol,LTP,Reference,Range Low,Range High,Distance (%),Change (%),Volume
0,NSE:INFY-EQ,1020.5,1000,950.0,1050.0,2.05,-0.86,5108734
1,NSE:TCS-EQ,2089.6,2000,1900.0,2100.0,4.48,-0.73,2228402


In [7]:
# ==========================================
# CELL 5 — LOAD NSE EQUITY UNIVERSE
# ==========================================

import pandas as pd

SYMBOL_MASTER_URL = "https://public.fyers.in/sym_details/NSE_CM.csv"

# Read raw FYERS symbol master
cm = pd.read_csv(
    SYMBOL_MASTER_URL,
    header=None
)

print("Rows:", len(cm))
print("Columns:", len(cm.columns))

# FYERS NSE CM:
# Column 3  (index 2)  = Instrument Type
# Column 10 (index 9)  = Symbol / trading symbol format
# Column 14 (index 13) = Underlying symbol

equity_master = cm[
    cm[2] == 0
].copy()

# Keep only normal NSE equity (EQ)
equity_master = equity_master[
    equity_master[9].astype(str).str.endswith("-EQ")
].copy()

# Create FYERS symbols
equity_master["fyers_symbol"] = equity_master[9].astype(str)

# Underlying stock name
equity_master["stock"] = equity_master[13].astype(str)

equity_master = equity_master[
    ["stock", "fyers_symbol"]
].drop_duplicates()

equity_master = equity_master.sort_values("fyers_symbol").reset_index(drop=True)

print("\nNSE equity stocks:", len(equity_master))

display(equity_master.head(20))

Rows: 10013
Columns: 21

NSE equity stocks: 2319


,stock,fyers_symbol
0,20MICRONS,NSE:20MICRONS-EQ
1,21STCENMGM,NSE:21STCENMGM-EQ
2,360ONE,NSE:360ONE-EQ
3,3BBLACKBIO,NSE:3BBLACKBIO-EQ
4,3MINDIA,NSE:3MINDIA-EQ
5,3PLAND,NSE:3PLAND-EQ
6,5PAISA,NSE:5PAISA-EQ
7,63MOONS,NSE:63MOONS-EQ
8,A2ZINFRA,NSE:A2ZINFRA-EQ
9,AAATECH,NSE:AAATECH-EQ


In [8]:
# ==========================================
# CELL 6 — FETCH LTP FOR ALL NSE STOCKS
# ==========================================

import time
import pandas as pd

BATCH_SIZE = 50

quote_results = []

symbols = equity_master["fyers_symbol"].tolist()

print(f"Total stocks : {len(symbols)}")
print(f"Batch size   : {BATCH_SIZE}")
print(f"Total batches: {(len(symbols) + BATCH_SIZE - 1) // BATCH_SIZE}")

for start in range(0, len(symbols), BATCH_SIZE):

    batch = symbols[start:start + BATCH_SIZE]

    data = {
        "symbols": ",".join(batch)
    }

    try:
        response_batch = fyers.quotes(data=data)

        if response_batch.get("s") == "ok":

            for item in response_batch.get("d", []):

                if item.get("s") != "ok":
                    continue

                quote = item.get("v", {})

                quote_results.append({
                    "Symbol": item.get("n"),
                    "LTP": quote.get("lp"),
                    "Change (%)": quote.get("chp"),
                    "Volume": quote.get("volume"),
                    "High": quote.get("high_price"),
                    "Low": quote.get("low_price"),
                    "Open": quote.get("open_price"),
                    "Prev Close": quote.get("prev_close_price"),
                })

        else:
            print(
                f"Batch {start // BATCH_SIZE + 1} failed:",
                response_batch
            )

    except Exception as e:
        print(
            f"Batch {start // BATCH_SIZE + 1} error:",
            e
        )

    # Small pause between API requests
    time.sleep(0.2)

quote_df = pd.DataFrame(quote_results)

print("\n--------------------------------")
print("QUOTE SCAN COMPLETE")
print("--------------------------------")
print("Stocks received:", len(quote_df))

display(quote_df.head(20))

Total stocks : 2319
Batch size   : 50
Total batches: 47

--------------------------------
QUOTE SCAN COMPLETE
--------------------------------
Stocks received: 2319


,Symbol,LTP,Change (%),Volume,High,Low,Open,Prev Close
0,NSE:20MICRONS-EQ,219.21,1.28,104573,223.95,215.80,220.80,219.21
1,NSE:21STCENMGM-EQ,39.92,0.76,6072,39.95,39.01,39.94,39.92
2,NSE:360ONE-EQ,1108.00,3.21,3642452,1122.00,1082.40,1093.70,1108.00
3,NSE:3BBLACKBIO-EQ,1431.60,4.55,6553,1440.00,1391.10,1400.00,1431.60
4,NSE:3MINDIA-EQ,34615.00,1.15,4992,35000.00,34225.00,34225.00,34615.00
5,NSE:3PLAND-EQ,30.91,0.55,2691,31.05,30.35,30.75,30.91
6,NSE:5PAISA-EQ,318.15,0.90,30838,321.70,315.00,317.25,318.15
7,NSE:63MOONS-EQ,852.95,1.33,510000,873.20,831.65,849.80,852.95
8,NSE:A2ZINFRA-EQ,14.79,0.41,192304,15.19,14.65,14.73,14.79
9,NSE:AAATECH-EQ,93.56,-0.18,116561,94.90,90.20,94.00,93.56


In [9]:
# ==========================================
# CELL 7 — PRICE LEVEL ±5% SCANNER
# ==========================================

PRICE_LEVELS = [100, 500, 1000, 1500, 2000 ]
RANGE_PERCENT = 5.0

scanner_results = []

for reference_price in PRICE_LEVELS:

    lower_price = reference_price * (1 - RANGE_PERCENT / 100)
    upper_price = reference_price * (1 + RANGE_PERCENT / 100)

    matches = quote_df[
        (quote_df["LTP"] >= lower_price) &
        (quote_df["LTP"] <= upper_price)
    ].copy()

    if not matches.empty:

        matches["Reference"] = reference_price

        matches["Range"] = (
            f"₹{lower_price:.2f} - ₹{upper_price:.2f}"
        )

        matches["Distance (%)"] = (
            (matches["LTP"] - reference_price)
            / reference_price
            * 100
        ).round(2)

        scanner_results.append(matches)


if scanner_results:

    price_scanner_df = pd.concat(
        scanner_results,
        ignore_index=True
    )

    price_scanner_df = price_scanner_df[
        [
            "Symbol",
            "LTP",
            "Reference",
            "Range",
            "Distance (%)",
            "Change (%)",
            "Volume"
        ]
    ]

    price_scanner_df = price_scanner_df.sort_values(
        ["Reference", "Distance (%)"]
    ).reset_index(drop=True)

    # Add serial number
    price_scanner_df.insert(
        0,
        "No.",
        range(1, len(price_scanner_df) + 1)
    )

    # Display complete list only once
    pd.set_option("display.max_rows", None)
    pd.set_option("display.max_columns", None)

    print(
        f"Stocks found within ±{RANGE_PERCENT}% "
        f"of selected price levels: "
        f"{len(price_scanner_df)}"
    )

    display(price_scanner_df)

else:

    print("No stocks found near the selected price levels.")

Stocks found within ±5.0% of selected price levels: 200


,No.,Symbol,LTP,Reference,Range,Distance (%),Change (%),Volume
0,1,NSE:VASUPRADA-EQ,95.05,100,₹95.00 - ₹105.00,-4.95,-4.95,31
1,2,NSE:UMIYA-MRO-EQ,95.10,100,₹95.00 - ₹105.00,-4.90,-2.25,11302
2,3,NSE:ANDHRSUGAR-EQ,95.20,100,₹95.00 - ₹105.00,-4.80,-0.16,213868
3,4,NSE:NTPCGREEN-EQ,95.42,100,₹95.00 - ₹105.00,-4.58,2.26,5223366
4,5,NSE:APCL-EQ,95.93,100,₹95.00 - ₹105.00,-4.07,1.19,2451
5,6,NSE:VISAKAIND-EQ,96.08,100,₹95.00 - ₹105.00,-3.92,1.70,149545
6,7,NSE:PREMIERPOL-EQ,96.72,100,₹95.00 - ₹105.00,-3.28,19.38,6517058
7,8,NSE:JAICORPLTD-EQ,96.73,100,₹95.00 - ₹105.00,-3.27,2.63,498373
8,9,NSE:AVTNPL-EQ,96.88,100,₹95.00 - ₹105.00,-3.12,-1.40,1064371
9,10,NSE:WCIL-EQ,97.33,100,₹95.00 - ₹105.00,-2.67,2.30,183868


In [10]:
# ==========================================
# CELL 7 — PRICE LEVEL ±5% SCANNER + CSV
# ==========================================

PRICE_LEVELS = [100, 500, 1000, 2000]
RANGE_PERCENT = 5.0

scanner_results = []

for reference_price in PRICE_LEVELS:

    lower_price = reference_price * (1 - RANGE_PERCENT / 100)
    upper_price = reference_price * (1 + RANGE_PERCENT / 100)

    matches = quote_df[
        (quote_df["LTP"] >= lower_price) &
        (quote_df["LTP"] <= upper_price)
    ].copy()

    if not matches.empty:

        matches["Reference"] = reference_price

        matches["Range"] = (
            f"₹{lower_price:.2f} - ₹{upper_price:.2f}"
        )

        matches["Distance (%)"] = (
            (matches["LTP"] - reference_price)
            / reference_price
            * 100
        ).round(2)

        scanner_results.append(matches)


if scanner_results:

    price_scanner_df = pd.concat(
        scanner_results,
        ignore_index=True
    )

    price_scanner_df = price_scanner_df[
        [
            "Symbol",
            "LTP",
            "Reference",
            "Range",
            "Distance (%)",
            "Change (%)",
            "Volume"
        ]
    ]

    price_scanner_df = price_scanner_df.sort_values(
        ["Reference", "Distance (%)"]
    ).reset_index(drop=True)

    # Add serial number
    price_scanner_df.insert(
        0,
        "No.",
        range(1, len(price_scanner_df) + 1)
    )

    # Display complete result
    pd.set_option("display.max_rows", None)
    pd.set_option("display.max_columns", None)

    print(
        f"Stocks found within ±{RANGE_PERCENT}% "
        f"of selected price levels: "
        f"{len(price_scanner_df)}"
    )

    display(price_scanner_df)

    # --------------------------------------
    # SAVE RESULT TO CSV
    # --------------------------------------

    CSV_PATH = PROJECT_DIR / "notebooks" / "price_scanner_results.csv"

    price_scanner_df.to_csv(
        CSV_PATH,
        index=False
    )

    print("\nCSV saved successfully:")
    print(CSV_PATH)

else:

    print("No stocks found near the selected price levels.")

Stocks found within ±5.0% of selected price levels: 168


,No.,Symbol,LTP,Reference,Range,Distance (%),Change (%),Volume
0,1,NSE:VASUPRADA-EQ,95.05,100,₹95.00 - ₹105.00,-4.95,-4.95,31
1,2,NSE:UMIYA-MRO-EQ,95.10,100,₹95.00 - ₹105.00,-4.90,-2.25,11302
2,3,NSE:ANDHRSUGAR-EQ,95.20,100,₹95.00 - ₹105.00,-4.80,-0.16,213868
3,4,NSE:NTPCGREEN-EQ,95.42,100,₹95.00 - ₹105.00,-4.58,2.26,5223366
4,5,NSE:APCL-EQ,95.93,100,₹95.00 - ₹105.00,-4.07,1.19,2451
5,6,NSE:VISAKAIND-EQ,96.08,100,₹95.00 - ₹105.00,-3.92,1.70,149545
6,7,NSE:PREMIERPOL-EQ,96.72,100,₹95.00 - ₹105.00,-3.28,19.38,6517058
7,8,NSE:JAICORPLTD-EQ,96.73,100,₹95.00 - ₹105.00,-3.27,2.63,498373
8,9,NSE:AVTNPL-EQ,96.88,100,₹95.00 - ₹105.00,-3.12,-1.40,1064371
9,10,NSE:WCIL-EQ,97.33,100,₹95.00 - ₹105.00,-2.67,2.30,183868



CSV saved successfully:
/home/manish/trading_project/stock_screener/notebooks/price_scanner_results.csv


In [11]:
# ==========================================
# CELL 8 — ABOVE PRICE LEVEL 10% SCANNER
#          + SAVE CSV
# ==========================================

PRICE_LEVELS = [100,200,300,500,1000,2000]
RANGE_PERCENT = 5.0

scanner_results = []

for reference_price in PRICE_LEVELS:

    lower_price = reference_price
    upper_price = reference_price * (1 + RANGE_PERCENT / 100)

    matches = quote_df[
        (quote_df["LTP"] >= lower_price) &
        (quote_df["LTP"] <= upper_price)
    ].copy()

    if not matches.empty:

        matches["Reference"] = reference_price

        matches["Range"] = (
            f"₹{lower_price:.2f} - ₹{upper_price:.2f}"
        )

        matches["Distance (%)"] = (
            (matches["LTP"] - reference_price)
            / reference_price
            * 100
        ).round(2)

        scanner_results.append(matches)


if scanner_results:

    price_scanner_df = pd.concat(
        scanner_results,
        ignore_index=True
    )

    price_scanner_df = price_scanner_df[
        [
            "Symbol",
            "LTP",
            "Reference",
            "Range",
            "Distance (%)",
            "Change (%)",
            "Volume"
        ]
    ]

    price_scanner_df = price_scanner_df.sort_values(
        ["Reference", "Distance (%)"],
        ascending=[True, True]
    ).reset_index(drop=True)

    price_scanner_df.insert(
        0,
        "No.",
        range(1, len(price_scanner_df) + 1)
    )

    pd.set_option("display.max_rows", None)
    pd.set_option("display.max_columns", None)

    print(
        f"Stocks above reference levels "
        f"within {RANGE_PERCENT}%: "
        f"{len(price_scanner_df)}"
    )

    display(price_scanner_df)

    # --------------------------------------
    # SAVE CSV
    # --------------------------------------

    CSV_PATH = (
        PROJECT_DIR
        / "notebooks"
        / "above_price_scanner_results.csv"
    )

    price_scanner_df.to_csv(
        CSV_PATH,
        index=False
    )

    print("\nCSV saved successfully:")
    print(CSV_PATH)

else:

    print("No stocks found above the selected price levels.")

Stocks above reference levels within 5.0%: 119


,No.,Symbol,LTP,Reference,Range,Distance (%),Change (%),Volume
0,1,NSE:INDORAMA-EQ,100.00,100,₹100.00 - ₹105.00,0.00,-3.45,1969705
1,2,NSE:SARLAPOLY-EQ,100.03,100,₹100.00 - ₹105.00,0.03,0.78,66846
2,3,NSE:ANSALBU-EQ,100.14,100,₹100.00 - ₹105.00,0.14,11.12,179538
3,4,NSE:ENIL-EQ,100.17,100,₹100.00 - ₹105.00,0.17,0.53,2106
4,5,NSE:JKIPL-EQ,100.25,100,₹100.00 - ₹105.00,0.25,1.02,3606
5,6,NSE:MAHEPC-EQ,100.45,100,₹100.00 - ₹105.00,0.45,1.72,18962
6,7,NSE:ATLASCYCLE-EQ,100.46,100,₹100.00 - ₹105.00,0.46,7.39,239595
7,8,NSE:DBREALTY-EQ,101.48,100,₹100.00 - ₹105.00,1.48,5.00,1033102
8,9,NSE:ALKALI-EQ,101.63,100,₹100.00 - ₹105.00,1.63,-1.52,169113
9,10,NSE:SAMBANDAM-EQ,102.50,100,₹100.00 - ₹105.00,2.50,1.49,1229



CSV saved successfully:
/home/manish/trading_project/stock_screener/notebooks/above_price_scanner_results.csv
